In [38]:
import os
import cv2
import glob
from PIL import Image
import numpy as np
import pickle

import torch
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset, SequentialSampler, random_split
import torch.nn.functional as F

import torch.nn as nn
import torch.optim as optim
from torchvision import datasets

from opacus.layers import DPLSTM
from opacus import PrivacyEngine
from opacus.utils.batch_memory_manager import BatchMemoryManager

import warnings
warnings.filterwarnings('ignore')

from util import cnnlstm

import uuid

Load the model

In [84]:
# Generate data for validation
from util import testdata_generate, train, cnnlstm, cnn3d, DPtrain

import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
batch_size = 1

path_validate = "test_split/Horse/HorseR0P0Y0_0"

# path_validate = './test_split'
test_dataset = testdata_generate.VideoFrameDataset(path_validate, 1)
test_loader = DataLoader(test_dataset)
print(test_loader)


In [85]:
for inputs, targets, masks, max_seq, position_intervals in test_loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            break

In [86]:
test_loader

In [87]:
inputs

tensor([[[[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]]]]], device='mps:0')

In [88]:
masks

tensor([[1.]], device='mps:0')

In [89]:
for data in test_loader:
    print(data)

[tensor([[[[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]]]]]), tensor([0]), tensor([[1.]]), tensor([22]), tensor([0])]
[tensor([[[[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]]]]]), tensor([0]), tensor([[1.]]), tensor([22]), tensor([1])]
[tensor([[[[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]]]]]), tensor([0]), tensor([[1.]]), tensor([22]), tenso

In [90]:
import pickle
import torch
from util import cnnlstm

model_name = "CNNLSTM"

# Load the model and tokenizer using pickle
with open('save/cnnlstm_e5_b8ed7038fcc611f09084bab6975ba98b.pkl', 'rb') as f:
    model = pickle.load(f)
model.to(device)
print(model)
print(f"Model {model_name} loaded successfully using pickle")

cnnlstm(
  (conv1): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(16, 16, eps=1e-05, affine=True)
  )
  (conv2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(16, 32, eps=1e-05, affine=True)
    (2): Dropout(p=0.3, inplace=False)
  )
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (lstm): LSTM(2048, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=10, bias=True)
)
Model CNNLSTM loaded successfully using pickle


In [91]:
def predict(model, loader):
    model.eval()
    predictions = []
    
    with torch.no_grad():
        # Iterate over the DataLoader, not the dataset
        for batch in loader: 
            # 1. Unpack the batch (Dataset returns 5 values)
            inputs, targets, masks, max_seq, position_intervals = batch
            
            # 2. Move tensors to the correct device (GPU/MPS/CPU)
            inputs = inputs.to(device)
            masks = masks.to(device)
            
            # 3. Pass ONLY the inputs and masks to the model
            # The error happened here previously because 'inputs' was a tuple
            outputs = model(inputs, masks)
            
            # Get predictions
            _, predicted = torch.max(outputs.data, 1)
            predictions.append(predicted)
            
    return predictions

In [92]:
output = predict(model, test_loader)
print(output)

[tensor([4], device='mps:0'), tensor([4], device='mps:0'), tensor([4], device='mps:0'), tensor([4], device='mps:0'), tensor([4], device='mps:0'), tensor([4], device='mps:0'), tensor([4], device='mps:0'), tensor([4], device='mps:0'), tensor([0], device='mps:0'), tensor([0], device='mps:0')]


In [ ]:
from collections import Counter

# Convert GPU tensors to standard Python integers
predicted_indices = [p.item() for p in output]

print(f"Segment Predictions: {predicted_indices}")

# Perform Majority Voting to find the most common class
counts = Counter(predicted_indices)
final_class_index = counts.most_common(1)[0][0]

print(f"Final Predicted Class Index: {final_class_index}")
class_names = ['Cup', 'FathersDay', 'Gun_like_objects', 'GunPart', 'Horse', 'Pistol', 'PortalGun', 'Revolver', 'SpecialRevolver', 'ToyGun'] 
print(f"Predicted Label: {class_names[final_class_index]}")

Segment Predictions: [4, 4, 4, 4, 4, 4, 4, 4, 0, 0]
Final Predicted Class Index: 4
Predicted Label: Horse
